## Experimento 4: Sustracción Espectral con STFT

**Hipótesis:** Configuraremos un factor de agresividad alto ($\alpha = 7.0$) para intentar reducir el mayor ruido posible, así mismo se aplicara una normalización gaussiana para intentar disminuir ese "ruido musical".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import stft, istft
import scipy.ndimage as ndimage


# 1. Cargar el audio
samplerate, data = wavfile.read('Audio1.wav')
if len(data.shape) > 1:
    data = data[:, 0]

# 2. Normalización de datos 
data_norm = data / np.max(np.abs(data))

In [ ]:
# 1. Aplicar la Transformada Rápida de Fourier por ventanas (STFT)
f, time_stft, Zxx = stft(data_norm, fs=samplerate, nperseg=2048)

# Separar el volumen (magnitud) y el retraso de la onda (fase)
magnitud = np.abs(Zxx)
fase = np.angle(Zxx)

# 2. Capturar la "huella dactilar" del ruido ambiental (usando el primer medio segundo)
tiempo_ruido_segundos = 0.5 
muestras_ruido = int(tiempo_ruido_segundos / (time_stft[1] - time_stft[0]))
perfil_ruido = np.mean(magnitud[:, :muestras_ruido], axis=1, keepdims=True)

In [ ]:
AGRESIVIDAD = 7.0
ruido_amplificado = perfil_ruido * AGRESIVIDAD

# 2. Cálculo de Relación Señal/Ruido (SNR) y creación de la Máscara
snr = magnitud / (ruido_amplificado + 1e-10)
mascara = 1.0 - (1.0 / snr)

# 3. Forzar un piso de ruido casi nulo
mascara[mascara < 0.001] = 0.001 

# 4. Suavizado matemático (elimina las "burbujas" o artefactos robóticos)
mascara_suavizada = ndimage.gaussian_filter(mascara, sigma=(1, 2))
magnitud_limpia = magnitud * mascara_suavizada


In [ ]:
import os

# 1. Unir la magnitud limpia con la fase original
Zxx_limpio = magnitud_limpia * np.exp(1j * fase)

# 2. Transformada Inversa (Regresamos al dominio del tiempo)
_, audio_recuperado = istft(Zxx_limpio, fs=samplerate)

# 3. Guardar el archivo usando un nombre incremental sin sobrescribir
base_nombre = 'Audio'
extension = '.wav'
contador = 1

while os.path.exists(f'{base_nombre}{contador}{extension}'):
    contador += 1

nombre_archivo = f'{base_nombre}{contador}{extension}'
audio_final = np.int16(audio_recuperado * 32767 / np.max(np.abs(audio_recuperado)))
wavfile.write(nombre_archivo, samplerate, audio_final)


**Resultados:** El "ruido musical" desaparecierón por completo. Esto valida empíricamente que el suavizado espacial de la matriz es la solución matemática correcta para las discontinuidades espectrales. Aunque el ruido fue erradicado en su totalidad, la agresividad de $\alpha = 7.0$ resultará excesiva. El algoritmo restará demasiada energía a las celdas que contienen frecuencias compartidas entre la voz y el ruido. Como resultado, la voz humana sufrirá una severa distorsión espectral, sonando sumamente opaca y sintética.